# Step 1 — Data Collection
**Project:** SDG Mapping for PhD Thesis  
**Purpose:** Download OSDG labeled dataset + load MAHE scraped corpus → save clean datasets for all downstream steps  
**Prerequisite:** Run `mahe_sdg_scraper.ipynb` first to produce `mahe_sdg_publications.csv`  
**Outputs:**
- `data/clean/osdg_clean.csv` — labeled training data
- `data/clean/mahe_corpus.csv` — unlabeled MAHE application corpus
- `data/step1_meta.json` — metadata for Steps 2–8


In [1]:
# Cell 1: Install dependencies 
!pip install pandas requests


In [2]:
# Cell 2: Imports & folder setup 
import io, json, requests
import pandas as pd
from pathlib import Path
from IPython.display import display

DATA_DIR  = Path("data")
RAW_DIR   = DATA_DIR / "raw"
CLEAN_DIR = DATA_DIR / "clean"
for d in [RAW_DIR, CLEAN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SDG_NAMES = {
    1:"No Poverty", 2:"Zero Hunger", 3:"Good Health and Well-Being",
    4:"Quality Education", 5:"Gender Equality", 6:"Clean Water and Sanitation",
    7:"Affordable and Clean Energy", 8:"Decent Work and Economic Growth",
    9:"Industry, Innovation and Infrastructure", 10:"Reduced Inequalities",
    11:"Sustainable Cities and Communities", 12:"Responsible Consumption and Production",
    13:"Climate Action", 14:"Life Below Water", 15:"Life on Land",
    16:"Peace, Justice and Strong Institutions", 17:"Partnerships for the Goals",
}

print("✅ Setup complete — folders created")


✅ Setup complete — folders created


In [5]:
# Cell 3: Download OSDG dataset 
# Primary source: Zenodo (https://zenodo.org/record/5550238)
# If download fails: manually download the CSV from Zenodo and place at
#   data/raw/osdg_community_raw.csv

OSDG_URL      = "https://zenodo.org/record/5550238/files/osdg-community-data-v2021-04-01.csv"
OSDG_RAW_PATH = RAW_DIR / "osdg_community_raw.csv"
OSDG_ALT_URL  = "https://raw.githubusercontent.com/osdg-ai/osdg-data/main/data/osdg-community-data.csv"

df_osdg_raw = None

if OSDG_RAW_PATH.exists():
    print(f"📦 Cache found at {OSDG_RAW_PATH} — loading...")
    df_osdg_raw = pd.read_csv(OSDG_RAW_PATH, sep="\t", on_bad_lines="skip")
    if len(df_osdg_raw.columns) < 3:
        df_osdg_raw = pd.read_csv(OSDG_RAW_PATH, on_bad_lines="skip")
else:
    for url in [OSDG_URL, OSDG_ALT_URL]:
        print(f"⬇ Downloading from {url} ...")
        try:
            r = requests.get(url, timeout=90)
            r.raise_for_status()
            sep = "\t" if "\t" in r.text[:500] else ","
            df_osdg_raw = pd.read_csv(io.StringIO(r.text), sep=sep, on_bad_lines="skip")
            df_osdg_raw.to_csv(OSDG_RAW_PATH, index=False)
            print(f"✅ Downloaded {len(df_osdg_raw):,} rows")
            break
        except Exception as e:
            print(f"  ⚠ Failed: {e}")

if df_osdg_raw is None:
    print("❌ Download failed. Place OSDG CSV manually at: data/raw/osdg_community_raw.csv")
else:
    print(f"\nShape: {df_osdg_raw.shape}")
    display(df_osdg_raw.head(3))


📦 Cache found at data\raw\osdg_community_raw.csv — loading...

Shape: (43025, 7)


,doi,text_id,text,sdg,labels_negative,labels_positive,agreement
0,10.6027/9789289342698-7-en,00021941702cd84171ff33962197ca1f,"From a gender perspective, Paulgaard points ou...",5,1,8,0.777778
1,10.18356/eca72908-en,00028349a7f9b2485ff344ae44ccfd6b,Labour legislation regulates maximum working h...,11,2,1,0.333333
2,10.1787/9789264289062-4-en,0004eb64f96e1620cd852603d9cbe4d4,The average figure also masks large difference...,3,1,8,0.777778


In [6]:
# Cell 4: Clean OSDG dataset 
df_osdg_raw.columns = [c.strip().lower().replace(" ", "_") for c in df_osdg_raw.columns]

# Identify text column
text_col = next((c for c in df_osdg_raw.columns if "text" in c and "id" not in c), None)
print(f"Text column detected: '{text_col}'")
print(f"All columns: {list(df_osdg_raw.columns)}")

# Agreement filter — keep majority-agreed labels only
if {"labels_positive", "labels_negative"}.issubset(df_osdg_raw.columns):
    total = df_osdg_raw["labels_positive"] + df_osdg_raw["labels_negative"]
    df_osdg_raw["agreement"] = df_osdg_raw["labels_positive"] / total.replace(0, 1)
    df_filtered = df_osdg_raw[df_osdg_raw["agreement"] >= 0.6].copy()
    print(f"\nAfter agreement >= 0.6 filter: {len(df_filtered):,} rows (from {len(df_osdg_raw):,})")
else:
    df_filtered = df_osdg_raw.copy()
    print("No agreement columns found — using all rows.")

# Build clean DataFrame
df_osdg = pd.DataFrame({
    "text":     df_filtered[text_col].astype(str).str.strip(),
    "sdg":      pd.to_numeric(df_filtered["sdg"], errors="coerce"),
    "sdg_name": pd.to_numeric(df_filtered["sdg"], errors="coerce").map(SDG_NAMES),
    "source":   "osdg",
})

df_osdg = df_osdg.dropna(subset=["text","sdg"])
df_osdg = df_osdg[df_osdg["sdg"].between(1,17)]
df_osdg["sdg"] = df_osdg["sdg"].astype(int)
df_osdg = df_osdg[df_osdg["text"].str.split().str.len() >= 10]
df_osdg = df_osdg.reset_index(drop=True)

print(f"\n✅ Clean OSDG: {len(df_osdg):,} rows | {df_osdg['sdg'].nunique()} SDGs")
display(df_osdg.head(3))


Text column detected: 'text'
All columns: ['doi', 'text_id', 'text', 'sdg', 'labels_negative', 'labels_positive', 'agreement']

After agreement >= 0.6 filter: 34,005 rows (from 43,025)

✅ Clean OSDG: 34,005 rows | 16 SDGs


,text,sdg,sdg_name,source
0,"From a gender perspective, Paulgaard points ou...",5,Gender Equality,osdg
1,The average figure also masks large difference...,3,Good Health and Well-Being,osdg
2,Applied research is directed “primarily toward...,9,"Industry, Innovation and Infrastructure",osdg


In [8]:
# Cell 5: Load MAHE corpus 
MAHE_CSV = Path("mahe_sdg_publications.csv")

if not MAHE_CSV.exists():
    print("⚠  mahe_sdg_publications.csv not found.")
    print("   Run mahe_sdg_scraper.ipynb first.")
    df_mahe = pd.DataFrame(columns=["text","sdg","sdg_name","source","title","url"])
else:
    raw_mahe     = pd.read_csv(MAHE_CSV)
    title_col    = next((c for c in raw_mahe.columns if "title"    in c.lower()), None)
    abstract_col = next((c for c in raw_mahe.columns if "abstract" in c.lower()), None)
    url_col      = next((c for c in raw_mahe.columns if "url"      in c.lower()
                                                     or "link"     in c.lower()), None)

    if title_col and abstract_col:
        raw_mahe["text"] = (raw_mahe[title_col].fillna("") + " " +
                            raw_mahe[abstract_col].fillna("")).str.strip()
    elif title_col:
        raw_mahe["text"] = raw_mahe[title_col].fillna("").str.strip()
        print("⚠  No abstract column — using title only.")
    else:
        raise ValueError("No title column found in MAHE CSV.")

    df_mahe = pd.DataFrame({
        "text":     raw_mahe["text"],
        "sdg":      None,
        "sdg_name": None,
        "source":   "mahe",
        "title":    raw_mahe[title_col] if title_col else "",
        "url":      raw_mahe[url_col]   if url_col   else "",
    })
    df_mahe = df_mahe[df_mahe["text"].str.split().str.len() >= 5].reset_index(drop=True)
    print(f"✅ MAHE corpus: {len(df_mahe):,} documents loaded")
    display(df_mahe.head(3))


✅ MAHE corpus: 854 documents loaded


,text,sdg,sdg_name,source,title,url
0,Analytical and bioanalytical HPLC method for s...,None,None,mahe,Analytical and bioanalytical HPLC method for s...,https://researcher.manipal.edu/en/publications...
1,‘Solving’ as a key course learning outcome (CL...,None,None,mahe,‘Solving’ as a key course learning outcome (CL...,https://researcher.manipal.edu/en/publications...
2,Remote Sensing: A Satellite-Based Advanced Geo...,None,None,mahe,Remote Sensing: A Satellite-Based Advanced Geo...,https://researcher.manipal.edu/en/publications...


In [9]:
# Cell 6: Dataset audit 
print("=" * 60)
print("  DATASET AUDIT")
print("=" * 60)

# OSDG stats
wlen = df_osdg["text"].str.split().str.len()
print(f"\n[A] OSDG Training Dataset")
print(f"    Rows            : {len(df_osdg):,}")
print(f"    SDGs covered    : {df_osdg['sdg'].nunique()} / 17")
print(f"    Avg text length : {wlen.mean():.0f} words")
print(f"    Min / Max       : {wlen.min()} / {wlen.max()} words")

print(f"\n    SDG distribution:")
dist = df_osdg["sdg"].value_counts().sort_index()
for sdg_num, count in dist.items():
    bar   = "█" * max(1, count * 30 // dist.max())
    label = SDG_NAMES.get(int(sdg_num), "?")[:36]
    print(f"    SDG {int(sdg_num):2d} | {bar:<30} | {count:5,}  {label}")

ratio = dist.max() / dist.min()
if ratio > 3:
    print(f"\n    ⚠  Imbalance ratio: {ratio:.1f}x — handle in Step 4 with class weights/SMOTE")

# MAHE stats
print(f"\n[B] MAHE Corpus")
if len(df_mahe) == 0:
    print("    (Empty — run scraper first)")
else:
    wlen_m = df_mahe["text"].str.split().str.len()
    print(f"    Documents       : {len(df_mahe):,}")
    print(f"    Avg text length : {wlen_m.mean():.0f} words")
    print(f"    Labels          : None (model will predict)")

print("\n" + "=" * 60)


  DATASET AUDIT

[A] OSDG Training Dataset
    Rows            : 34,005
    SDGs covered    : 16 / 17
    Avg text length : 95 words
    Min / Max       : 16 / 226 words

    SDG distribution:
    SDG  1 | ██████████████                 | 1,983  No Poverty
    SDG  2 | ███████████                    | 1,611  Zero Hunger
    SDG  3 | █████████████████              | 2,382  Good Health and Well-Being
    SDG  4 | ██████████████████████         | 3,080  Quality Education
    SDG  5 | ███████████████████████        | 3,255  Gender Equality
    SDG  6 | ██████████████                 | 2,076  Clean Water and Sanitation
    SDG  7 | ██████████████████             | 2,550  Affordable and Clean Energy
    SDG  8 | █████████                      | 1,308  Decent Work and Economic Growth
    SDG  9 | ████████████████               | 2,352  Industry, Innovation and Infrastruct
    SDG 10 | ████████████                   | 1,710  Reduced Inequalities
    SDG 11 | ██████████████                 | 1,

In [10]:
# Cell 7: Save clean datasets 
osdg_out = CLEAN_DIR / "osdg_clean.csv"
mahe_out = CLEAN_DIR / "mahe_corpus.csv"

df_osdg.to_csv(osdg_out, index=False, encoding="utf-8-sig")
print(f"💾 Saved: {osdg_out}")

if len(df_mahe) > 0:
    df_mahe.to_csv(mahe_out, index=False, encoding="utf-8-sig")
    print(f"💾 Saved: {mahe_out}")

meta = {
    "osdg_rows":    len(df_osdg),
    "mahe_rows":    len(df_mahe),
    "sdg_names":    SDG_NAMES,
    "osdg_path":    str(osdg_out),
    "mahe_path":    str(mahe_out),
    "label_counts": {str(k): int(v) for k,v in df_osdg["sdg"].value_counts().sort_index().items()},
}
meta_path = DATA_DIR / "step1_meta.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)
print(f"💾 Saved: {meta_path}")

print("\n✅ Step 1 complete → next: step2_eda.ipynb")


💾 Saved: data\clean\osdg_clean.csv
💾 Saved: data\clean\mahe_corpus.csv
💾 Saved: data\step1_meta.json

✅ Step 1 complete → next: step2_eda.ipynb
